FROM TRY HARD GUIDES

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from time import sleep

In [ ]:
BASE_URL = "https://tryhardguides.com"
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/91.0.4472.124 Safari/537.36'
    )
}

In [ ]:
def get_article_links(page_number):
    """
    Fetch all daily NYT Strands article links from a given tag page.
    """
    url = f"{BASE_URL}/tag/nyt-strands/page/{page_number}/"
    print(f"Scraping article list from: {url}")
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"  > Could not fetch page {page_number}. Error: {e}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")
    links = []

    # Typical article listing selector
    for header in soup.select("article.post-listing h2.entry-title a"):
        link = header.get("href")
        if link and "nyt-strands-answers" in link:
            links.append(link)

    return links

In [ ]:
def parse_article_page(url):
    print(f"  > Parsing article: {url.split('/')[-2]}")
    try:
        response = requests.get(url, headers=HEADERS, timeout=12)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"    > Could not fetch article. Error: {e}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")
    article_content = soup.find("div", class_="entry-content")
    if not article_content:
        return None

    # --- Basic info ---
    title = soup.find("h1", class_="entry-title").get_text(strip=True) if soup.find("h1") else "N/A"
    date = soup.find("time", class_="entry-date").get("datetime") if soup.find("time") else "N/A"

    # --- Extract words & spangram ---
    theme_line = "N/A"
    spangram = "N/A"
    words = []

    answers_header = article_content.find("h2", string=re.compile(r"Answers$", re.IGNORECASE))
    if answers_header:
        answer_list = answers_header.find_next_sibling("ul")
        if answer_list:
            for item in answer_list.find_all("li"):
                text = item.get_text(strip=True)
                if "(Spangram)" in text:
                    spangram = text.replace("(Spangram)", "").strip()
                else:
                    words.append(text)

    # --- Extract FULL THEME LINE ---
    all_text_nodes = article_content.find_all(string=True)
    for node in all_text_nodes:
        text = node.strip()
        if not text:
            continue

        # Normalize smart quotes/apostrophes for easier matching
        normalized = (
            text.replace("“", '"')
                .replace("”", '"')
                .replace("‘", "'")
                .replace("’", "'")
        )

        # Look for a line that starts with "Today's theme is" or "Today’s theme is"
        if re.search(r"today.?s\s+theme\s+is", normalized, re.IGNORECASE):
            theme_line = text.strip()
            break

    # --- Extract GRID ---
    grid_letters = []
    puzzle_figure = soup.find("figure", class_="nyt-strands-puzzle")
    if puzzle_figure:
        for letter_wrap in puzzle_figure.select(".letter-wrap div"):
            letter = letter_wrap.get_text(strip=True)
            if letter:
                grid_letters.append(letter)
    else:
        puzzle_div = soup.find("div", class_="nyt-strands-puzzle")
        if puzzle_div:
            for letter_wrap in puzzle_div.select(".letter-wrap div"):
                letter = letter_wrap.get_text(strip=True)
                if letter:
                    grid_letters.append(letter)

    return {
        "title": title,
        "url": url,
        "date": date,
        "theme_line": theme_line,
        "spangram": spangram,
        "words": ", ".join(words),
        "grid": "".join(grid_letters)
    }

In [ ]:
def build_dataset(total_pages=30, delay=1):
    all_data = []

    for page in range(1, total_pages + 1):
        links = get_article_links(page)
        if not links:
            print(f"No more articles found on page {page}. Stopping.")
            break

        for link in links:
            article_data = parse_article_page(link)
            if article_data:
                all_data.append(article_data)
            sleep(delay)

    df = pd.DataFrame(all_data)
    df.to_csv("tryhardguides_strands.csv", index=False, encoding="utf-8-sig")
    print(f"\n✓ Saved {len(df)} articles to tryhardguides_strands.csv")
    return df

In [ ]:
if __name__ == "__main__":
    df = build_dataset(total_pages=30, delay=1.5)

    print("\n=== SAMPLE FROM tryhardguides_strands.csv (First 5 Rows) ===")
    print(df.head().to_string(index=False))

Scraping article list from: https://tryhardguides.com/tag/nyt-strands/page/1/
  > Parsing article: nyt-strands-answers-november-6-2025
  > Parsing article: nyt-strands-answers-november-5-2025
  > Parsing article: nyt-strands-answers-november-4-2025
  > Parsing article: nyt-strands-answers-november-2-2025
  > Parsing article: nyt-strands-answers-november-1-2025
  > Parsing article: nyt-strands-answers-october-31-2025
  > Parsing article: nyt-strands-answers-october-30-2025
  > Parsing article: nyt-strands-answers-october-29-2025
  > Parsing article: nyt-strands-answers-october-28-2025
  > Parsing article: nyt-strands-answers-october-27-2025
  > Parsing article: nyt-strands-answers-october-26-2025
  > Parsing article: nyt-strands-answers-october-25-2025
  > Parsing article: nyt-strands-answers-october-24-2025
  > Parsing article: nyt-strands-answers-october-23-2025
  > Parsing article: nyt-strands-answers-october-22-2025
  > Parsing article: nyt-strands-answers-october-21-2025
  > Parsin